# Option A – patch run (v2): fairness check, 5 seeds, longer ST-SHN

This notebook **adds to** the first run (`optionA_outputs.zip`, v1). It does not repeat the city forecasts, the intervals or the rolling origins; it reads their saved predictions. It runs only what was missing or too thin in v1:

| Part | What runs | Why |
|---|---|---|
| **V. Equity (census)** | ACS 2022 tract data with a Census API key → poverty and minority share per area → coverage, skill and risk lists by group | v1: the Census API refused the request without a key, so A8 was not evaluated |
| **I.2 Re-runs, more seeds** | ST-HSL, Graph WaveNet, MTGNN: seeds 3 and 4 added to v1's seeds 0–2. Our benchmark forecaster: seeds 0–4 | 5 seeds, so the NYC reproduction gap can be tested against seed noise |
| **I.2 ST-SHN** | 5 seeds × 2 cities, 25 epochs, predictions saved after every epoch | v1: only 2 seeds and the official 10 epochs; one run's best epoch was its last one. The first 10 epochs of a 25-epoch run are the official setting (the learning-rate schedule depends on steps, not on the epoch total), so both budgets come from the same runs |
| **I.3 + VI** | Reproduction gap (now with a 95 % t-interval per cell), metric exploit, rankings, intervals, all verdicts | recomputed on the merged predictions |

The main ST-SHN rows keep the **official budget of 10 epochs** (as published); the 25-epoch budget is reported separately in `b16_stshn_epoch_budget.csv` as a sensitivity check.

**Before you run (one time)**
1. **Previous outputs as input.** *Add Input → Datasets → Your datasets* → the dataset made from `optionA_outputs.zip` (v1). The notebook finds `tables/b06_benchmark_scores_per_seed.csv` anywhere under `/kaggle/input`.
2. **Census API key.** Get a free key at https://api.census.gov/data/key_signup.html and activate it from the e-mail. Then *Add-ons → Secrets → Add secret*: label `CENSUS_API_KEY`, value = the key, and switch it on for this notebook.
3. *Settings → Accelerator → GPU T4 x2*, *Internet → On*.
4. **Save Version → Save & Run All (Commit).** Expected run time 7–8.5 hours.

**If the run stops early** (time limit or error): every finished re-run is saved at once in `predictions/patch_runs/`. Make a dataset from the patch output zip, add it as a second input, and run again: finished runs are skipped.

**What to send back:** `optionA_outputs_v2.zip` from the *Output* tab. It holds the **complete** v1 + v2 result set (v1's logs are kept in `logs_v1/`).

## Setup
Same configuration and helpers as v1 (the output zip is now `optionA_outputs_v2.zip`), followed by the patch settings: seeds, ST-SHN epoch budget, and the v1 outputs copied in.

In [ ]:
import os, sys, time, json, math, gc, re, warnings, zipfile, shutil, subprocess, random, traceback, hashlib, io, contextlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
T0 = time.time()

# ------------------------------------------------------------------ configuration
FAST        = os.environ.get("OA_FAST", "0") == "1"          # quick smoke test only
TIME_LIMIT_H = 11.4                                           # stop optional work before Kaggle's 12-hour limit
START_YEAR  = int(os.environ.get("OA_START_YEAR", "2016"))
TRAIN_END   = pd.Timestamp("2023-12-31")                      # same split as the capstone
VAL_END     = pd.Timestamp("2024-06-30")                      # calibration / validation block; test = after this date
CITIES      = ["CHI", "NYC", "SF"]
CITY_NAME   = {"CHI": "Chicago", "NYC": "New York", "SF": "San Francisco"}
DROP_LAST_WEEKS = {"CHI": 1, "NYC": 4, "SF": 2}               # late-reported records make the newest weeks incomplete
HORIZONS    = [1, 2, 3, 4]
NN_HORIZONS = [1, 4]                                          # neural forecasters (slower) are run for these horizons
SEEDS_TREE  = [0] if FAST else [0, 1, 2]
SEEDS_NN    = [0] if FAST else [0, 1, 2]
SEEDS_BENCH = [0] if FAST else [0, 1, 2]                      # ST-HSL, Graph WaveNet, MTGNN, our benchmark forecaster
SEEDS_STSHN = [0] if FAST else [0, 1]                         # ST-SHN (TensorFlow, batch size 1, slower)
ROLL_YEARS  = [2021, 2022] if FAST else [2020, 2021, 2022, 2023, 2024]
ALPHA       = 0.10                                            # 90 % intervals
SEED        = 42

KAGGLE = Path("/kaggle/working").exists()
WORK   = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OA_WORK", Path.cwd()))
OUT    = WORK / "optionA_outputs"; FIG = OUT / "figures"; TAB = OUT / "tables"; LOGS = OUT / "logs"; PRED = OUT / "predictions"
DATA   = WORK / "raw_data"; REPOS = WORK / "repos"
for p in (OUT, FIG, TAB, LOGS, PRED, DATA, REPOS): p.mkdir(parents=True, exist_ok=True)

def hours_used(): return (time.time() - T0) / 3600
def time_left(): return TIME_LIMIT_H - hours_used()
def log(msg):
    line = f"[{hours_used():5.2f} h] {msg}"; print(line, flush=True)
    with open(LOGS / "progress.log", "a") as f: f.write(line + "\n")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=140, bbox_inches="tight"); plt.show(); plt.close("all")
def save_table(df, name, index=False):
    df.to_csv(TAB / f"{name}.csv", index=index); return df
def refresh_zip():
    """Zip everything in the output folder (called after every part)."""
    dest = WORK / "optionA_outputs_v2.zip"; tmp = WORK / "optionA_outputs_v2.zip.part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file(): z.write(p, "optionA_outputs/" + str(p.relative_to(OUT)))
    tmp.replace(dest); log(f"zip refreshed: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

STATUS = {}
@contextlib.contextmanager
def part(name):
    """Run one part; on an error write the traceback to logs/errors.log and continue with the next part."""
    t = time.time(); log(f"===== {name} started"); STATUS[name] = "running"
    try:
        yield; STATUS[name] = f"ok ({(time.time() - t) / 60:.0f} min)"
    except Exception:
        tb = traceback.format_exc(); STATUS[name] = "FAILED – see logs/errors.log"
        print(tb); open(LOGS / "errors.log", "a").write(f"===== {name}\n{tb}\n")
    finally:
        log(f"===== {name}: {STATUS[name]}"); json.dump(STATUS, open(LOGS / "status.json", "w"), indent=1)
        try: refresh_zip()
        except Exception as e: print("zip failed:", e)

def pip_install(*pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True, timeout=900)
try: import lightgbm as lgb
except ImportError: pip_install("lightgbm"); import lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy.stats import poisson as _pois, nbinom as _nb, kendalltau, spearmanr

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

def pick_device():
    if torch.cuda.is_available():
        try: torch.ones(8, device="cuda").sum().item(); return torch.device("cuda")
        except Exception as e: print("CUDA present but unusable with this PyTorch build:", str(e)[:120])
    return torch.device("cpu")
DEVICE = pick_device()
def xgb_device():
    try:
        b = xgb.XGBRegressor(n_estimators=2, device="cuda", tree_method="hist"); b.fit(np.random.rand(64, 3), np.random.rand(64)); return "cuda"
    except Exception: return "cpu"
XGB_DEVICE = xgb_device() if DEVICE.type == "cuda" else "cpu"
VERSIONS = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__, xgboost=xgb.__version__,
                lightgbm=lgb.__version__, device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", fast=FAST)
json.dump(VERSIONS, open(LOGS / "environment.json", "w"), indent=1)
log(f"environment: {VERSIONS}")
if DEVICE.type == "cpu" and not FAST: print("WARNING: no usable GPU. Switch the accelerator to 'GPU T4 x2', or expect a much longer run.")

# ------------------------------------------------------------------ shared metric helpers
def pdev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * np.mean(np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def unit_dev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def block_boot(diff, blocks, B=1000, seed=SEED):
    """Mean of `diff` and a 95 % CI from a bootstrap that resamples whole blocks (weeks or days)."""
    wk = pd.factorize(np.asarray(blocks))[0]; s = np.bincount(wk, diff); n = np.bincount(wk); r = np.random.default_rng(seed)
    idx = r.integers(0, len(s), (B, len(s))); est = s[idx].sum(1) / n[idx].sum(1)
    return float(np.mean(diff)), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

# ------------------------------------------------------------------ shared forecasters (gradient-boosted Poisson models)
def fit_lgb_pois(Xtr, ytr, Xes, yes, seed=0, n_est=None):
    m = lgb.LGBMRegressor(objective="poisson", n_estimators=n_est or (150 if FAST else 3000), learning_rate=0.05, num_leaves=127, min_child_samples=20,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_xgb_pois(Xtr, ytr, Xes, yes, seed=0):
    m = xgb.XGBRegressor(objective="count:poisson", n_estimators=150 if FAST else 2000, learning_rate=0.05, max_depth=7, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=5, tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                         max_cat_to_onehot=1, random_state=seed, early_stopping_rounds=50, eval_metric="poisson-nloglik")
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False); return m
def fit_lgb_quantile(Xtr, ytr, Xes, yes, q, seed=0):
    m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_lgb_l2(Xtr, ytr, Xes, yes, seed=0):
    m = lgb.LGBMRegressor(objective="l2", n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m

# ------------------------------------------------------------------ shared interval helpers (all calibrated on a held-out block only)
def conformal_q(scores, alpha=ALPHA):
    scores = np.asarray(scores, float); n = len(scores)
    if n == 0: return np.inf
    lev = np.ceil((n + 1) * (1 - alpha)) / n
    return np.inf if lev > 1 else float(np.quantile(scores, lev, method="higher"))
def norm_score(y, mu): return np.abs(y - mu) / np.sqrt(mu + 0.5)
def int_from_q(mu, q, scale=None):
    s = np.sqrt(mu + 0.5) if scale is None else scale
    lo = np.maximum(np.ceil(mu - q * s), 0); hi = np.floor(mu + q * s)
    bad = hi < lo; r = np.round(mu); return np.where(bad, r, lo), np.where(bad, r, hi)
def mondrian_edges(mu_cal, k=5): return np.unique(np.quantile(mu_cal, np.linspace(0, 1, k + 1)[1:-1]))
def nb_size(mu, y):                                            # negative-binomial size by moments: Var = mu + mu^2 / r
    return float(min(max(np.sum(mu ** 2) / max(np.sum((y - mu) ** 2 - mu), 1e-6), 1e-3), 1e4))
def parametric_interval(mu, kind, r=None):
    mu = np.maximum(mu, 1e-6)
    if kind == "Poisson": return _pois.ppf(ALPHA / 2, mu), _pois.ppf(1 - ALPHA / 2, mu)
    p = r / (r + mu); return _nb.ppf(ALPHA / 2, r, p), _nb.ppf(1 - ALPHA / 2, r, p)
def interval_score(y, lo, hi): return (hi - lo) + (2 / ALPHA) * np.maximum(lo - y, 0) + (2 / ALPHA) * np.maximum(y - hi, 0)
def coverage_report(y, lo, hi, vq=None, area=None):
    """Overall coverage, width, interval score, coverage per volume quintile and per area."""
    cov = (y >= lo) & (y <= hi); out = dict(coverage=cov.mean(), width=(hi - lo).mean(), interval_score=interval_score(y, lo, hi).mean())
    if vq is not None:
        for q in range(1, 6): out[f"cov_Q{q}"] = cov[vq == q].mean() if (vq == q).any() else np.nan
        out["max_abs_dev_from_90"] = np.nanmax([abs(out[f"cov_Q{q}"] - (1 - ALPHA)) for q in range(1, 6)])
        out["gap_Q5_minus_Q1"] = out["cov_Q5"] - out["cov_Q1"]
    if area is not None:
        ac = pd.Series(cov).groupby(np.asarray(area)).mean()
        out["areas_within_5pts"] = float((ac.sub(1 - ALPHA).abs() <= 0.05).mean()); out["worst_area_dev"] = float(ac.sub(1 - ALPHA).abs().max())
        out["sd_area_coverage"] = float(ac.std())
    return out

In [ ]:
# ------------------------------------------------------------------ patch (v2) configuration
SEEDS_BENCH  = [0] if FAST else [0, 1, 2, 3, 4]               # all benchmark models end with 5 seeds
NEW_SEEDS    = [] if FAST else [3, 4]                         # seeds 0-2 of ST-HSL / Graph WaveNet / MTGNN come from v1
SEEDS_STSHN  = [0] if FAST else [0, 1, 2, 3, 4]               # ST-SHN: every seed is re-run (v1 had 2 seeds, 10 epochs)
STSHN_EPOCHS = 2 if FAST else 25                              # longer budget (sensitivity check)
STSHN_OFFICIAL_EPOCHS = 2 if FAST else 10                     # official default of the ST-SHN code (Params.py)
INPUT_ROOT   = Path(os.environ.get("OA_INPUT", "/kaggle/input"))

def find_prev():
    """Folder of the v1 outputs (the one that contains tables/b06_benchmark_scores_per_seed.csv)."""
    hits = sorted(INPUT_ROOT.rglob("b06_benchmark_scores_per_seed.csv")) if INPUT_ROOT.exists() else []
    hits = [h for h in hits if not (h.parent.parent / "logs_v1").exists()]       # skip the output of an earlier patch session
    if not hits: raise FileNotFoundError("v1 outputs not found under /kaggle/input: add the dataset made from optionA_outputs.zip (Add Input).")
    if len(hits) > 1: print("several v1 output folders found, using the first:", [str(h) for h in hits])
    return hits[0].parent.parent
PREV = find_prev(); log(f"v1 outputs: {PREV}")

# copy v1 into the output folder, so the final zip is the complete result set (v2 tables overwrite v1 ones)
for sub in ("tables", "figures", "predictions"):
    if (PREV / sub).exists(): shutil.copytree(PREV / sub, OUT / sub, dirs_exist_ok=True)
for f in ("results_summary.md", "metrics_summary.json"):
    if (PREV / f).exists() and not (OUT / f"{Path(f).stem}_v1{Path(f).suffix}").exists(): shutil.copy(PREV / f, OUT / f"{Path(f).stem}_v1{Path(f).suffix}")
if (PREV / "logs").exists(): shutil.copytree(PREV / "logs", OUT / "logs_v1", dirs_exist_ok=True)
STATUS_V1 = json.load(open(PREV / "logs" / "status.json")) if (PREV / "logs" / "status.json").exists() else {}

# finished re-runs from an earlier, interrupted patch session (added as a second input) are reused
RUNS = PRED / "patch_runs"; RUNS.mkdir(parents=True, exist_ok=True)
for f in (INPUT_ROOT.rglob("patch_runs/*") if INPUT_ROOT.exists() else []):
    if f.is_file() and not (RUNS / f.name).exists(): shutil.copy(f, RUNS / f.name)
log(f"finished re-runs already available: {len(list(RUNS.glob('*.npz')))}")

def tab_v1(name, **kw):
    """A v1 table (read from the copied v1 folder)."""
    p = PREV / "tables" / f"{name}.csv"
    return pd.read_csv(p, **kw) if p.exists() and p.stat().st_size > 2 else None

class _NpzDir(dict):
    @property
    def files(self): return list(self.keys())
def load_npz(name):
    """v1 .npz file; Kaggle may unpack it into a folder of .npy files when the dataset is uploaded."""
    for root in (PREV / "predictions", PRED):
        f = root / name
        if f.is_file(): return np.load(f)
        if f.is_dir(): return _NpzDir({p.stem: np.load(p) for p in f.rglob("*.npy")})
    raise FileNotFoundError(f"{name} not found in the v1 outputs (predictions/)")

## Part I.1 Benchmark code, data and protocol audit (as in v1)
Clones the four official repositories (the commits are logged in `b01`; compare them with v1) and repeats the code audit.

In [ ]:
BENCH_CATS = {"NYC": ["Burglary", "Larceny", "Robbery", "Assault"], "CHI": ["Theft", "Battery", "Assault", "Damage"]}
BENCH_CITIES = ["NYC", "CHI"]
# Table III of Li et al. (2022, ICDE), ST-HSL paper: MAE, MAPE per category. Verify against the paper before quoting.
PUBLISHED = {
    "NYC": {"ARIMA": [0.8999, 0.6305, 1.3015, 0.6268, 0.9558, 0.5969, 0.9983, 0.6198], "SVM": [1.1604, 0.7653, 1.4979, 0.6417, 1.1278, 0.6733, 1.1928, 0.6964],
            "ST-ResNet": [0.8680, 0.5603, 1.1082, 0.5329, 0.8717, 0.5209, 0.9645, 0.5749], "DCRNN": [0.8176, 0.5324, 1.0732, 0.5492, 0.9189, 0.5532, 0.9692, 0.5955],
            "STGCN": [0.8366, 0.5404, 1.0629, 0.5295, 0.9035, 0.5441, 0.9375, 0.5757], "GWN": [0.7993, 0.5235, 1.0493, 0.5405, 0.8681, 0.5351, 0.8866, 0.5646],
            "STtrans": [0.8617, 0.5592, 1.0896, 0.5478, 0.8839, 0.5651, 0.9363, 0.5679], "DeepCrime": [0.8227, 0.5508, 1.0618, 0.5351, 0.8841, 0.5537, 0.9222, 0.5677],
            "MTGNN": [0.8329, 0.5439, 1.0473, 0.5330, 0.8759, 0.5457, 0.9090, 0.5714], "ST-SHN": [0.8012, 0.5198, 1.0431, 0.5291, 0.8717, 0.5362, 0.9169, 0.5682],
            "DMSTGCN": [0.8376, 0.5485, 1.0410, 0.5464, 0.8597, 0.5403, 0.9036, 0.5601], "ST-HSL": [0.7329, 0.4788, 1.0316, 0.5040, 0.7912, 0.4595, 0.8484, 0.5029]},
    "CHI": {"ARIMA": [1.5965, 0.5720, 1.3212, 0.5792, 0.8691, 0.6044, 1.0430, 0.6134], "SVM": [1.7711, 0.5629, 1.3493, 0.6027, 1.0879, 0.6560, 1.1313, 0.5721],
            "ST-ResNet": [1.3931, 0.5488, 1.1519, 0.5719, 0.7679, 0.4633, 0.9064, 0.5018], "DCRNN": [1.3699, 0.5770, 1.1583, 0.5528, 0.7639, 0.4600, 0.8764, 0.4756],
            "STGCN": [1.3628, 0.5359, 1.1512, 0.5761, 0.7963, 0.4810, 0.9068, 0.4959], "GWN": [1.3211, 0.5502, 1.1331, 0.5503, 0.7493, 0.4580, 0.8584, 0.4850],
            "STtrans": [1.3404, 0.5356, 1.1466, 0.5684, 0.7671, 0.4499, 0.8987, 0.4842], "DeepCrime": [1.3391, 0.5430, 1.1290, 0.5389, 0.7737, 0.4616, 0.9096, 0.4960],
            "ST-MetaNet": [1.3369, 0.5369, 1.1762, 0.5748, 0.7904, 0.4753, 0.8907, 0.4756], "MTGNN": [1.3054, 0.5378, 1.1307, 0.5597, 0.7571, 0.4572, 0.8667, 0.4859],
            "ST-SHN": [1.3231, 0.5310, 1.1348, 0.5544, 0.7758, 0.4574, 0.8741, 0.4747], "DMSTGCN": [1.3292, 0.5291, 1.1297, 0.5552, 0.8058, 0.4759, 0.8698, 0.4877],
            "ST-HSL": [1.2952, 0.4929, 1.1016, 0.5231, 0.6665, 0.3996, 0.8446, 0.4644]}}
REPO_URLS = {"STHSL": "https://github.com/LZH-YS1998/STHSL", "ST-SHN": "https://github.com/akaxlh/ST-SHN",
             "Graph-WaveNet": "https://github.com/nnzhan/Graph-WaveNet", "MTGNN": "https://github.com/nnzhan/MTGNN"}
PATCHES = {"STHSL": "engine.py: running loss stored with .item() (memory only); train.py: device and seed read from arguments/environment (official default seed 523 = our seed 0)",
           "ST-SHN": "TensorFlow 1 code run with tf.compat.v1; xavier_initializer → glorot_uniform (same initialiser); softmax on 6-D tensor written out (numerically identical); seed from environment; predictions and validation metrics logged every epoch",
           "Graph-WaveNet": "nn.Conv1d with 2-D kernels replaced by nn.Conv2d (old PyTorch accepted this silently; identical computation)",
           "MTGNN": "none"}

def git_clone(name):
    dest = REPOS / name
    if not dest.exists():
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URLS[name], str(dest)], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(f"could not clone {name}: {r.stderr[-300:]}")
    h = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip()
    return dest, h

def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
def bench_load(c):
    import pickle
    t = {s: np.asarray(pickle.load(open(REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", "rb")), np.float32) for s in ("trn", "val", "tst")}
    return np.concatenate([t["trn"], t["val"], t["tst"]], 2), t["trn"].shape[2], t["val"].shape[2], t["tst"].shape[2]

from pathlib import Path
def patch_stshn(root):
    """Minimal patches so the official TF1 ST-SHN code runs on TF2 (compat.v1), plus logging hooks.
    Model, loss, data handling and the official metrics are unchanged."""
    root = Path(root)
    nn_ = root / "Utils" / "NNLayers.py"; s = nn_.read_text()
    if "compat.v1" not in s:
        s = s.replace("import tensorflow as tf\nfrom tensorflow.contrib.layers import xavier_initializer",
                      "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nxavier_initializer = tf.glorot_uniform_initializer", 1)
        assert "compat.v1" in s, "NNLayers import pattern not found"; nn_.write_text(s)
    m = root / "HG_ST_labcode.py"; s = m.read_text()
    if "compat.v1" in s: return
    rep = [
        ("import tensorflow as tf\n", "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nimport os as _os, json as _json\n"
         "def _softmax(x, axis):\n\tx = x - tf.reduce_max(x, axis=axis, keepdims=True); e = tf.exp(x)\n\treturn e / tf.reduce_sum(e, axis=axis, keepdims=True)\n"),
        # hook 1: keep the per-cell predictions of every evaluation pass (aligned with the day index)
        ("\t\tnum = len(ids)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)",
         "\t\tnum = len(ids)\n\t\tself._P = np.zeros([args.areaNum, tstTensor.shape[1], args.offNum], np.float32)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)"),
        ("\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n",
         "\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n\t\t\t\tself._P[:, batIds[0], :] = preds\n"),
        # hook 2: after every epoch evaluate the validation block too, and store validation metrics + test predictions
        ("\t\t\tif test:\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n",
         "\t\t\tif test:\n\t\t\t\tvres = self.testEpoch(self.handler.valT, self.handler.trnT)\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n\t\t\t\tself._hist.append(dict(epoch=ep, val=vres, test=reses)); self._preds.append(self._P.copy())\n\t\t\t\tnp.save(_os.environ.get('STSHN_OUT', 'stshn') + '_preds.npy', np.stack(self._preds)); _json.dump([{k: {kk: float(vv) for kk, vv in v.items()} if isinstance(v, dict) else v for k, v in h.items()} for h in self._hist], open(_os.environ.get('STSHN_OUT', 'stshn') + '_hist.json', 'w'))\n"),
        ("\t\tbestRes = None\n\t\tfor ep in range(stloc, args.epoch):", "\t\tbestRes = None\n\t\tself._hist, self._preds = [], []\n\t\tfor ep in range(stloc, args.epoch):"),
        # seed from the environment (the official code sets none)
        ("\tconfig = tf.ConfigProto()\n", "\t_sd = int(_os.environ.get('STSHN_SEED', '0')); np.random.seed(_sd); tf.set_random_seed(_sd)\n\tconfig = tf.ConfigProto()\n"),
    ]
    for a, b in rep:
        assert a in s, "ST-SHN patch pattern not found: " + a[:60]
        s = s.replace(a, b, 1)
    assert s.count("att = tf.nn.softmax(") == 2, "ST-SHN softmax pattern not found"
    s = s.replace("att = tf.nn.softmax(", "att = _softmax(")    # TF2 kernels reject softmax on 6-D tensors; same softmax written out
    m.write_text(s)

with part("I.1 benchmark: code, data and protocol audit"):
    VERS = []
    for n in REPO_URLS:
        try: d, h = git_clone(n); VERS.append(dict(repository=n, url=REPO_URLS[n], commit=h, patches=PATCHES[n]))
        except Exception as e: VERS.append(dict(repository=n, url=REPO_URLS[n], commit="NOT AVAILABLE", patches=str(e)[:200]))
    VERS = save_table(pd.DataFrame(VERS), "b01_code_versions_and_patches"); display(VERS)
    BENCH_OK = (REPOS / "STHSL" / "Datasets").exists()
    # same data in both official repositories?
    same = []
    for c in BENCH_CITIES:
        for s in ("trn", "val", "tst"):
            a, b = REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", REPOS / "ST-SHN" / "Datasets" / f"{c}_crime" / f"{s}.pkl"
            same.append(dict(city=c, split=s, sthsl_md5=md5(a) if a.exists() else None, stshn_md5=md5(b) if b.exists() else None))
    same = pd.DataFrame(same); same["identical"] = same["sthsl_md5"] == same["stshn_md5"]; save_table(same, "b02_dataset_identity"); display(same)

    # automatic audit of the evaluation code (run on the untouched clones, before any patch)
    def code_has(repo, file, pattern):
        p = REPOS / repo / file
        return bool(p.exists() and re.search(pattern, p.read_text()))
    AUDIT = [
        ("ST-HSL", "scored cell-days", "only cell-days with at least one crime (mask = label > 0)", code_has("STHSL", "engine.py", r"mask = 1 \* \(label > 0\)")),
        ("ST-HSL", "training loss", "squared error on every cell-day (mask = label >= 0)", code_has("STHSL", "engine.py", r"retLabels = \(label >= 0\) \* 1")),
        ("ST-HSL", "reported epoch", "best validation RMSE + MAE", code_has("STHSL", "train.py", r"val_metrics = res_eval\['RMSE'\] \+ res_eval\['MAE'\]")),
        ("ST-HSL", "random seed", "fixed (seed_torch, 523)", code_has("STHSL", "utils.py", r"def seed_torch\(seed=523\)")),
        ("ST-SHN", "scored cell-days", "crime days only, and only cells with crime on 20–80 % of training days",
         code_has("ST-SHN", "HG_ST_labcode.py", r"mask = self\.handler\.tstLocs \* \(label > 0\)") and code_has("ST-SHN", "DataHandler.py", r"\(percent > 0\.2\) \* \(percent < 0\.8\)")),
        ("ST-SHN", "training loss", "squared error on crime days only (regression: negRate = 0)", code_has("ST-SHN", "HG_ST_labcode.py", r"negRate = 0")),
        ("ST-SHN", "reported epoch", "epoch with the best TEST MAPE (the test set chooses the epoch)",
         code_has("ST-SHN", "HG_ST_labcode.py", r"reses = self\.testEpoch\(self\.handler\.tstT.*\n.*bestRes\['MAPE'\] > reses\['MAPE'\]")),
        ("ST-SHN", "random seed", "none set", not code_has("ST-SHN", "HG_ST_labcode.py", r"set_random_seed|np\.random\.seed")),
    ]
    AUDIT = save_table(pd.DataFrame(AUDIT, columns=["code", "aspect", "what the official code does", "verified in code"]), "b03_protocol_audit"); display(AUDIT)

    # how much of the test set does the official metric ignore?
    BSTAT = []
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; tst = Xall[:, :, ntr + nva:].reshape(R * Cc, nte, K)
        act = (Xall[:, :, :ntr] > 0).mean(2).reshape(R * Cc, K); tl = (act > 0.2) & (act < 0.8)
        for k, n in enumerate(BENCH_CATS[c]):
            y = tst[:, :, k]
            BSTAT.append(dict(city=c, category=n, cells=R * Cc, test_days=nte, mean_daily_count=y.mean(), share_zero_cell_days=(y == 0).mean(),
                              share_scored_P1=(y > 0).mean(), cells_in_P2_subset=int(tl[:, k].sum()), share_scored_P2=((y > 0) & tl[:, k][:, None]).mean()))
    BSTAT = save_table(pd.DataFrame(BSTAT), "b04_benchmark_sparsity"); display(BSTAT.round(3))

### Benchmark models (unchanged from v1)

In [ ]:
def bench_features(X, days):
    """X: (rows, cols, days, K). One row per (day, cell, category) in the order used by the official code."""
    R, Cc, T, K = X.shape; d = np.asarray(days)
    cs = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(X, 2)], 2)
    Pd = np.pad(X, ((1, 1), (1, 1), (0, 0), (0, 0))); NB = sum(Pd[1 + dy:1 + dy + R, 1 + dx:1 + dx + Cc] for dy in (-1, 0, 1) for dx in (-1, 0, 1)) - X
    csn = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(NB, 2)], 2)
    tot = X.sum(3, keepdims=True); cst = np.concatenate([np.zeros((R, Cc, 1, 1), np.float32), np.cumsum(tot, 2)], 2)
    f = {f"lag{l}": X[:, :, d - l] for l in range(1, 8)}
    for w in (7, 14, 30): f[f"m{w}"] = (cs[:, :, d] - cs[:, :, d - w]) / w
    f["same_dow4"] = np.mean([X[:, :, d - 7 * j] for j in (1, 2, 3, 4)], 0)
    f["nb1"] = NB[:, :, d - 1]; f["nb7"] = (csn[:, :, d] - csn[:, :, d - 7]) / 7; f["nb30"] = (csn[:, :, d] - csn[:, :, d - 30]) / 30
    f["cell_all7"] = np.broadcast_to((cst[:, :, d] - cst[:, :, d - 7]) / 7, f["lag1"].shape)
    f["city_lag1"] = np.broadcast_to(X.sum((0, 1), keepdims=True)[:, :, d - 1], f["lag1"].shape)
    df = pd.DataFrame({k: np.ascontiguousarray(np.moveaxis(v, 2, 0)).reshape(-1) for k, v in f.items()})
    n = len(d); df["cell"] = pd.Categorical(np.tile(np.repeat(np.arange(R * Cc), K), n)); df["cat"] = pd.Categorical(np.tile(np.arange(K), n * R * Cc))
    df["dow"] = np.repeat(d % 7, R * Cc * K)
    return df, np.moveaxis(X[:, :, d], 2, 0).reshape(-1)
def ztp_median(mu):
    mu = np.maximum(mu, 1e-3); p0 = np.exp(-mu); out = np.ones_like(mu)
    for k in range(1, 80): out = np.where((_pois.cdf(k, mu) - p0) / (1 - p0) < 0.5, k + 1, out)
    return out

# ---------------------------------------------------------------- ST-HSL (official code)
def prep_sthsl():
    d = REPOS / "STHSL"; eg = (d / "engine.py").read_text()
    if "epochLoss += loss\n" in eg: (d / "engine.py").write_text(eg.replace("epochLoss += loss\n", "epochLoss += loss.item()\n", 1))
    tr = (d / "train.py").read_text()
    if "torch.device(args.device)" not in tr: tr = tr.replace("torch.device('cuda:0')", "torch.device(args.device)")
    if "STHSL_SEED" not in tr:
        assert "    seed_torch()\n" in tr, "ST-HSL seed pattern not found"
        tr = tr.replace("    seed_torch()\n", "    import os as _os; seed_torch(int(_os.environ.get('STHSL_SEED', '523')))\n", 1)
    (d / "train.py").write_text(tr)
    (d / "dump_preds.py").write_text('''
import numpy as np, torch
from Params import args
from DataHandler import DataHandler
from model import STHSL
import engine
handler = DataHandler(); model = STHSL().to(args.device)
model.load_state_dict(torch.load(args.checkpoint, map_location=args.device)); model.eval()
def run(T, inp):
    ids = np.arange(T.shape[1]); P = []
    with torch.no_grad():
        for st in range(0, len(ids), args.batch):
            b = ids[st:st + args.batch]
            feats, labels, mask = engine.sampleTestBatch(b, st, st + len(b), T, inp, handler)
            feats = torch.Tensor(feats).to(args.device)
            out = model(feats, feats[:, np.random.permutation(args.areaNum)])[-1]
            P.append(handler.zInverse(out).cpu().numpy())
    return np.concatenate(P)
np.save("sthsl_pred_tst.npy", run(handler.tstT, np.concatenate([handler.trnT, handler.valT], axis=1)))
np.save("sthsl_pred_val.npy", run(handler.valT, handler.trnT))
''')

def run_logged(cmd, cwd, logfile, timeout_s, env=None):
    with open(logfile, "w") as lf:
        p = subprocess.Popen(cmd, cwd=cwd, stdout=lf, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
        try: rc = p.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired: p.kill(); rc = "timeout"
    return rc

def run_sthsl(c, seed, epochs, timeout_s):
    d = REPOS / "STHSL"; (d / "Save" / c).mkdir(parents=True, exist_ok=True)
    for f in (d / "Save" / c).glob("*.pth"): f.unlink()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    dev = "cuda:0" if DEVICE.type == "cuda" else "cpu"; t = time.time(); lf = LOGS / f"sthsl_{c}_seed{seed}.log"
    rc = run_logged([sys.executable, "train.py", "--data", c, "--device", dev, "--epoch", str(epochs), "--save", "./Save/"], d, lf, timeout_s,
                    env={"STHSL_SEED": str(523 if seed == 0 else seed)})
    txt = lf.read_text(errors="ignore").replace("\r", "\n"); ep, best_ep, n_ep = 0, None, 0
    for line in txt.splitlines():
        mm = re.match(r"Epoch\s+(\d+) Training Time", line.strip())
        if mm: ep = int(mm.group(1)); n_ep = ep
        if "Val metrics decrease" in line: best_ep = ep
    if best_ep is None: log(f"  ST-HSL {c} seed {seed}: no finished epoch (exit {rc}); see {lf.name}"); return None
    ck = sorted((d / "Save" / c).glob(f"_epoch_{best_ep}_MAE_*.pth"))
    if not ck: log(f"  ST-HSL {c} seed {seed}: checkpoint of epoch {best_ep} missing"); return None
    rc2 = run_logged([sys.executable, "dump_preds.py", "--data", c, "--device", dev, "--checkpoint", str(ck[0])], d, LOGS / f"sthsl_{c}_seed{seed}_dump.log", 1800)
    if rc2 != 0: log(f"  ST-HSL {c} seed {seed}: prediction dump failed ({rc2})"); return None
    info = dict(model="ST-HSL", city=c, seed=seed, epochs_run=n_ep, epochs_planned=epochs, selected_epoch=best_ep, selection="validation (official rule)",
                minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-HSL {c} seed {seed}: {n_ep}/{epochs} epochs, validation-best epoch {best_ep}, {info['minutes']:.0f} min")
    # (days, cells, K) predictions
    return np.load(d / "sthsl_pred_tst.npy"), np.load(d / "sthsl_pred_val.npy"), info

# ---------------------------------------------------------------- ST-SHN (official TensorFlow 1 code)
def prep_stshn():
    src, dst = REPOS / "ST-SHN", REPOS / "ST-SHN-run"
    if not dst.exists(): shutil.copytree(src, dst)
    patch_stshn(dst); (dst / "History").mkdir(exist_ok=True); (dst / "Models").mkdir(exist_ok=True)
    return dst
def run_stshn(c, seed, timeout_s, epochs=None):
    d = REPOS / "ST-SHN-run"; t = time.time(); lf = LOGS / f"stshn_{c}_seed{seed}.log"; outp = str(d / f"out_{c}_s{seed}")
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    cmd = [sys.executable, "HG_ST_labcode.py", "--data", c, "--task", "r", "--save_path", f"{c}_s{seed}"] + (["--epoch", str(epochs)] if epochs else [])
    env = {"STSHN_OUT": outp, "STSHN_SEED": str(seed), "TF_CPP_MIN_LOG_LEVEL": "2"}
    if DEVICE.type != "cuda": env["CUDA_VISIBLE_DEVICES"] = ""
    rc = run_logged(cmd, d, lf, timeout_s, env)
    if not Path(outp + "_preds.npy").exists(): log(f"  ST-SHN {c} seed {seed}: no epoch finished (exit {rc}); see {lf.name}"); return None
    P = np.load(outp + "_preds.npy"); H = json.load(open(outp + "_hist.json"))            # P: (epochs, cells, days, K)
    vm = np.array([h["val"]["MAPE"] for h in H]); tm = np.array([h["test"]["MAPE"] for h in H])
    e_val, e_tst = int(np.argmin(vm)), int(np.argmin(tm))
    info = dict(model="ST-SHN", city=c, seed=seed, epochs_run=len(H), epochs_planned=epochs or 10, selected_epoch_validation=e_val + 1,
                selected_epoch_test=e_tst + 1, minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-SHN {c} seed {seed}: {len(H)} epochs, epoch by validation {e_val + 1}, by test (official) {e_tst + 1}, {info['minutes']:.0f} min")
    return np.transpose(P[e_val], (1, 0, 2)), np.transpose(P[e_tst], (1, 0, 2)), info

# ---------------------------------------------------------------- Graph WaveNet / MTGNN (official model code, ST-HSL training rule)
def load_module(path, name, extra_path=None):
    import importlib.util
    if extra_path and str(extra_path) not in sys.path: sys.path.insert(0, str(extra_path))
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
def grid_adj(R, Cc):
    N = R * Cc; A = np.eye(N, dtype=np.float32)
    for i in range(R):
        for j in range(Cc):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if 0 <= i + di < R and 0 <= j + dj < Cc: A[i * Cc + j, (i + di) * Cc + (j + dj)] = 1
    return A
def make_graph_model(kind, N, K, R, Cc):
    if kind == "Graph WaveNet":
        gd = REPOS / "Graph-WaveNet"; mp = gd / "model_patched.py"
        if not mp.exists(): mp.write_text((gd / "model.py").read_text().replace("nn.Conv1d(", "nn.Conv2d("))
        gw = load_module(mp, "gwn_model_patched")
        A = grid_adj(R, Cc); P = A / A.sum(1, keepdims=True)
        sup = [torch.tensor(P, device=DEVICE), torch.tensor(P.T.copy(), device=DEVICE)]
        return gw.gwnet(DEVICE, N, dropout=0.3, supports=sup, gcn_bool=True, addaptadj=True, aptinit=None, in_dim=K, out_dim=K,
                        residual_channels=32, dilation_channels=32, skip_channels=256, end_channels=512).to(DEVICE)
    mt = load_module(REPOS / "MTGNN" / "net.py", "mtgnn_net", REPOS / "MTGNN")
    return mt.gtnet(True, True, 2, N, DEVICE, predefined_A=None, static_feat=None, dropout=0.3, subgraph_size=min(20, N), node_dim=40,
                    dilation_exponential=1, conv_channels=32, residual_channels=32, skip_channels=64, end_channels=128, seq_length=30,
                    in_dim=K, out_dim=K, layers=3, propalpha=0.05, tanhalpha=3, layer_norm_affline=True).to(DEVICE)
def train_graph_model(kind, c, seed, max_epochs=2 if FAST else 100, patience=10, batch=16, L=30):
    set_seed(seed); Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
    S = Xall.reshape(N, T, K); mean, std = float(Xall[:, :, :ntr].mean()), float(Xall[:, :, :ntr].std())
    Z = torch.tensor((S - mean) / std, device=DEVICE); Y = torch.tensor(S, device=DEVICE)
    model = make_graph_model(kind, N, K, R, Cc); opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    back = torch.arange(L, 0, -1, device=DEVICE)
    def fwd(days):
        win = days[:, None] - back[None]                                        # (B, L): days d-L .. d-1
        x = Z[:, win, :].permute(1, 3, 0, 2)                                    # (B, K, N, L)
        out = model(x)[..., -1]                                                 # (B, K, N)
        return out.permute(0, 2, 1) * std + mean                                # (B, N, K) counts
    def predict(days):
        model.eval(); out = []
        with torch.no_grad():
            for s in range(0, len(days), 32): out.append(fwd(days[s:s + 32]).cpu().numpy())
        return np.concatenate(out)
    trd = torch.arange(L, ntr, device=DEVICE); vad = torch.arange(ntr, ntr + nva, device=DEVICE); ted = torch.arange(ntr + nva, T, device=DEVICE)
    yv = S[:, ntr:ntr + nva].transpose(1, 0, 2); best, best_state, bad, t = np.inf, None, 0, time.time()
    for ep in range(max_epochs):
        model.train(); perm = trd[torch.randperm(len(trd), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = F.mse_loss(fwd(b), Y[:, b, :].permute(1, 0, 2))
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        pv = predict(vad); pos = yv > 0; e = pv - yv
        crit = np.sqrt((e[pos] ** 2).mean()) + np.abs(e[pos]).mean()           # ST-HSL rule: validation RMSE + MAE on crime days
        if crit < best - 1e-5: best, bad, best_state, best_ep = crit, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep + 1
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    info = dict(model=kind, city=c, seed=seed, epochs_run=ep + 1, epochs_planned=max_epochs, selected_epoch=best_ep, selection="validation (ST-HSL rule)",
                minutes=(time.time() - t) / 60, finished=True)
    log(f"  {kind} {c} seed {seed}: {ep + 1} epochs, best {best_ep}, {info['minutes']:.1f} min")
    return predict(ted), predict(vad), info

# ---------------------------------------------------------------- metrics under the three protocols
def p_metrics(p, y, cat, K, keep):
    """MAE / MAPE per category on the cell-days in `keep` (the official codes use keep = y > 0, ST-SHN also restricts cells)."""
    out = {}
    for k in range(K):
        m = keep & (cat == k); out[f"MAE_{k}"] = np.abs(p - y)[m].mean(); out[f"MAPE_{k}"] = (np.abs(p - y) / np.maximum(y, 1e-8))[m].mean()
    out["MAE"] = np.abs(p - y)[keep].mean(); out["MAPE"] = (np.abs(p - y) / np.maximum(y, 1e-8))[keep].mean(); return out
def full_metrics(p, y, cat, K):
    out = dict(MAE_all=np.abs(p - y).mean(), RMSE_all=np.sqrt(((p - y) ** 2).mean()), deviance_all=pdev(y, np.maximum(p, 0.01)),
               MAE_zero_days=np.abs(p - y)[y == 0].mean(), MAE_crime_days=np.abs(p - y)[y > 0].mean())
    for k in range(K): out[f"deviance_{k}"] = pdev(y[cat == k], np.maximum(p[cat == k], 0.01)); out[f"MAE_all_{k}"] = np.abs(p - y)[cat == k].mean()
    return out

## Part V. Equity check with census data (fixed)
Same method as v1 (ACS 2022 5-year, tract level; each tract goes to the area of the majority of its 25 nearest geolocated incidents; tracts more than 1 km from any incident are dropped). Changes: the request carries the **Census API key** from the Kaggle secret `CENSUS_API_KEY`; the error text of a refused request is logged; hand-supplied files (`acs2022_<state><county>.csv`, `gaz_tracts_<state>.txt`) anywhere under `/kaggle/input` are used if present; the raw census files are saved in `census/`. The incidents for the tract → area mapping (2024–2025) are downloaded again, and the intervals and forecasts are read from v1's saved predictions.

In [ ]:
SOURCES = {
    "CHI": dict(api="https://data.cityofchicago.org/resource/ijzp-q8t2.csv", date="date", first_year=2001,
                cols="date,primary_type,community_area,latitude,longitude"),
    "NYC": dict(api="https://data.cityofnewyork.us/resource/qgea-i56i.csv", date="cmplnt_fr_dt", first_year=2006,
                cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "NYC_YTD": dict(api="https://data.cityofnewyork.us/resource/5uac-w243.csv", date="cmplnt_fr_dt", first_year=2006,
                    cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "SF": dict(api="https://data.sfgov.org/resource/wg3w-h783.csv", date="incident_datetime", first_year=2018,
               cols="incident_datetime,incident_id,incident_category,analysis_neighborhood,latitude,longitude"),
}
CITY_CSV = {c: os.environ.get(f"OA_{c}_CSV", "") for c in CITIES}       # optional: your own extract with the API column names

def soda_download(key, dest, start_year, end_year=None, page=500_000, select=True):
    import urllib.request, urllib.parse, urllib.error
    src = SOURCES[key]; tmp = Path(str(dest) + ".part"); total, wrote_header = 0, False
    end_year = end_year or pd.Timestamp.today().year
    with open(tmp, "wb") as out:
        for yr in range(max(start_year, src["first_year"]), end_year + 1):
            offset, size, n_year = 0, page, 0
            while True:
                q = ({"$select": src["cols"]} if select else {}) | {"$where": f"{src['date']} between '{yr}-01-01T00:00:00' and '{yr}-12-31T23:59:59'",
                     "$order": ":id", "$limit": size, "$offset": offset}
                url = src["api"] + "?" + urllib.parse.urlencode(q); data, last = None, None
                for attempt in range(5):
                    try:
                        with urllib.request.urlopen(url, timeout=900) as r: data = r.read()
                        break
                    except urllib.error.HTTPError as e:
                        last = e
                        if e.code == 400: break                       # bad query (e.g. a renamed column): no point retrying
                        time.sleep(15 * (attempt + 1))
                    except Exception as e:
                        last = e; time.sleep(15 * (attempt + 1))
                if data is None: raise RuntimeError(f"{key}: request failed for {yr}: {last!r}")
                head, _, body = data.partition(b"\n"); n = body.count(b"\n") + (0 if (body.endswith(b"\n") or not body) else 1)
                if not wrote_header: out.write(head.strip() + b"\n"); wrote_header = True
                if body: out.write(body if body.endswith(b"\n") else body + b"\n")
                n_year += n; offset += n
                if n == 0: break
                if n < size:
                    if n in (1000, 50000): size = n; continue
                    break
            log(f"  {key} {yr}: {n_year:,} rows"); total += n_year
    if total == 0: raise RuntimeError(f"{key}: the portal returned no rows")
    tmp.replace(dest); return total
def get_file(key):
    if CITY_CSV.get(key) and Path(CITY_CSV[key]).exists(): return Path(CITY_CSV[key])
    dest = DATA / f"{key.lower()}_{START_YEAR}.csv"
    if not dest.exists():
        log(f"{key}: downloading from {START_YEAR} ...")
        try: soda_download(key, dest, START_YEAR)
        except RuntimeError as e:
            if "400" not in str(e): raise
            log(f"{key}: column selection refused by the portal ({str(e)[:120]}); downloading all columns instead"); soda_download(key, dest, START_YEAR, select=False)
    return dest


BOX = {"CHI": (41.60, 42.10, -87.95, -87.50), "NYC": (40.45, 40.95, -74.30, -73.65), "SF": (37.65, 37.85, -122.55, -122.33)}

In [ ]:
ACS_VARS = {"B01003_001E": "pop", "B17001_001E": "pov_universe", "B17001_002E": "pov", "B03002_001E": "race_total", "B03002_003E": "nh_white", "B19013_001E": "median_income"}
COUNTIES = {"CHI": [("17", "031")], "NYC": [("36", "005"), ("36", "047"), ("36", "061"), ("36", "081"), ("36", "085")], "SF": [("06", "075")]}
GEO_YEARS = (2024, 2025)                                       # geolocated incidents used to map census tracts to areas
CENSUS_OUT = OUT / "census"; CENSUS_OUT.mkdir(exist_ok=True)   # raw census files are kept, so the check can be repeated offline

def census_key():
    try:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret("CENSUS_API_KEY")
        if k: return k.strip()
    except Exception: pass
    return os.environ.get("CENSUS_API_KEY", "").strip()
CENSUS_KEY = census_key()
log(f"Census API key: {'found' if CENSUS_KEY else 'NOT found – add the secret CENSUS_API_KEY (Add-ons → Secrets)'}")

def from_input(name):
    """A census file supplied by hand (any dataset under /kaggle/input), if there is one."""
    hits = sorted(INPUT_ROOT.rglob(name)) if INPUT_ROOT.exists() else []
    return hits[0] if hits else None
def http_get(url, timeout=120):
    import urllib.request, urllib.error
    last = None
    for a in range(4):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r: return r.read()
        except urllib.error.HTTPError as e:
            last = e
            try: last = RuntimeError(f"HTTP {e.code}: {e.read()[:300]!r}")
            except Exception: pass
            if e.code in (400, 401, 403, 404): break
            time.sleep(5 * (a + 1))
        except Exception as e: last = e; time.sleep(5 * (a + 1))
    raise RuntimeError(f"{url.split('&key=')[0]}: {last!r}")
def acs_tracts(state, county):
    name = f"acs2022_{state}{county}.csv"; f = CENSUS_OUT / name
    if not f.exists() and from_input(name): shutil.copy(from_input(name), f)
    if not f.exists():
        key = f"&key={CENSUS_KEY}" if CENSUS_KEY else ""; base = f"https://api.census.gov/data/2022/acs/acs5?get={','.join(ACS_VARS)}&for=tract:*"
        errs = []
        for geo in (f"&in=state:{state}&in=county:{county}", f"&in=state:{state}%20county:{county}"):
            try: raw = http_get(base + geo + key)
            except Exception as e: errs.append(str(e)[:300]); continue
            if raw.lstrip()[:1] != b"[": errs.append("not JSON: " + raw[:300].decode("utf-8", "replace").replace("\n", " ")); continue
            js = json.loads(raw); pd.DataFrame(js[1:], columns=js[0]).to_csv(f, index=False); break
        else: raise RuntimeError("Census API: " + " | ".join(errs))
    d = pd.read_csv(f, dtype=str); d["GEOID"] = d["state"].str.zfill(2) + d["county"].str.zfill(3) + d["tract"].str.zfill(6)
    for v, n in ACS_VARS.items(): d[n] = pd.to_numeric(d[v], errors="coerce")
    d.loc[d["median_income"] < 0, "median_income"] = np.nan
    return d[["GEOID"] + list(ACS_VARS.values())]
def gazetteer(state):
    name = f"gaz_tracts_{state}.txt"; f = CENSUS_OUT / name
    if not f.exists() and from_input(name): shutil.copy(from_input(name), f)
    if not f.exists():
        last = None
        for url in (f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2022_Gazetteer/2022_gaz_tracts_{state}.txt",
                    f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2023_Gazetteer/2023_gaz_tracts_{state}.txt",
                    f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_gaz_tracts_{state}.txt"):
            try: f.write_bytes(http_get(url)); break
            except Exception as e: last = e
        else: raise RuntimeError(f"gazetteer for state {state} not available: {last}")
    g = pd.read_csv(f, sep="\t", dtype=str); g.columns = [c_.strip() for c_ in g.columns]
    return pd.DataFrame({"GEOID": g["GEOID"].str.zfill(11), "lat": pd.to_numeric(g["INTPTLAT"]), "lon": pd.to_numeric(g["INTPTLONG"])})

def geo_points(c):
    """Geolocated incidents of GEO_YEARS with their area label (same area rules as Part II.1 of v1)."""
    dest = DATA / f"{c.lower()}_geo_{GEO_YEARS[0]}_{GEO_YEARS[1]}.csv"
    if not dest.exists(): log(f"{c}: downloading {GEO_YEARS[0]}–{GEO_YEARS[1]} incidents for the tract → area mapping ..."); soda_download(c, dest, GEO_YEARS[0], GEO_YEARS[1])
    df = pd.read_csv(dest, dtype=str)
    if c == "CHI":
        area = pd.to_numeric(df["community_area"], errors="coerce"); area = area.where((area >= 1) & (area <= 77))
    elif c == "NYC":
        area = pd.to_numeric(df["addr_pct_cd"], errors="coerce").replace({116: 105}); area = area.where((area >= 1) & (area <= 123))
    if c in ("CHI", "NYC"): area = area.astype("Int64").astype(str).where(area.notna())
    else:
        df = df.drop_duplicates(subset=["incident_id", "incident_category"])
        area = df["analysis_neighborhood"].where(df["analysis_neighborhood"].notna() & (df["analysis_neighborhood"].str.strip() != ""))
    lat = pd.to_numeric(df["latitude"], errors="coerce"); lon = pd.to_numeric(df["longitude"], errors="coerce"); b = BOX[c]
    g = pd.DataFrame({"lat": lat, "lon": lon, "area": area})[lat.between(b[0], b[1]) & lon.between(b[2], b[3]) & area.notna()]
    return g.sample(min(len(g), 300_000), random_state=SEED).reset_index(drop=True)
def tracts_to_areas(c, tr):
    from sklearn.neighbors import BallTree
    P = GEO[c]; tree = BallTree(np.radians(P[["lat", "lon"]].values.astype(float)), metric="haversine")
    dist, idx = tree.query(np.radians(tr[["lat", "lon"]].values), k=25); lab = np.asarray(P["area"].astype(str), dtype=object)[idx]
    tr = tr.copy(); tr["area"] = [pd.Series(r).mode().iloc[0] for r in lab]; tr["nearest_km"] = dist[:, 0] * 6371.0
    return tr[tr["nearest_km"] <= 1.0]

# saved v1 predictions replace the in-memory objects of Parts II–III
def find_pred(name):
    """A v1 prediction file. Kaggle may unpack `x.csv.gz` into `x.csv` (or into a folder) when a dataset is uploaded, so every form is tried."""
    for root in (PRED, PREV / "predictions"):
        for cand in (root / name, root / name.removesuffix(".gz")):
            if cand.is_file(): return cand
            if cand.is_dir():
                inner = [p for p in cand.rglob("*") if p.is_file()]
                if inner: return inner[0]
    hits = [p for p in INPUT_ROOT.rglob(name.removesuffix(".gz") + "*") if p.is_file()] if INPUT_ROOT.exists() else []
    return hits[0] if hits else None
def read_pred(name):
    f = find_pred(name)
    if f is None: raise FileNotFoundError(f"{name} not found in the v1 outputs (predictions/). Files there: {sorted(p.name for p in (PREV / 'predictions').iterdir())}")
    log(f"  reading {f}"); return pd.read_csv(f, dtype={"area": str})

IV, GEO, AREAS = {}, {}, {}
WK1 = read_pred("weekly_test_predictions_H1.csv.gz")
for c in CITIES:
    if find_pred(f"intervals_test_H1_{c}.csv.gz"): IV[c] = read_pred(f"intervals_test_H1_{c}.csv.gz"); AREAS[c] = sorted(IV[c]["area"].unique())
CITIES_OK = [c for c in CITIES if c in IV]
SCOL = "lgbm_s0"                                               # selected weekly forecaster (v1: LightGBM-Poisson), seed 0, H = 1

DEMO, EQ_COV, EQ_SKILL, EQ_RISK, AREA_EQ = [], [], [], [], []
with part("V. equity check (census)"):
    for c in CITIES_OK:
        try:
            tr = pd.concat([acs_tracts(s, k) for s, k in COUNTIES[c]]).merge(pd.concat([gazetteer(s) for s in sorted({s for s, _ in COUNTIES[c]})]), on="GEOID")
            GEO[c] = geo_points(c)                                     # census first: a refused census request fails fast, before the incident download
            tr = tracts_to_areas(c, tr[tr["pop"] > 0])
        except Exception as e:
            log(f"  census data for {c} not available: {str(e)[:400]}"); continue
        g = tr.groupby("area").apply(lambda d: pd.Series(dict(tracts=len(d), pop=d["pop"].sum(), poverty_rate=d["pov"].sum() / max(d["pov_universe"].sum(), 1),
                                                             minority_share=1 - d["nh_white"].sum() / max(d["race_total"].sum(), 1),
                                                             median_income=np.average(d["median_income"].fillna(d["median_income"].median()), weights=d["pop"]))))
        g = g.reset_index().assign(city=c); g["poverty_group"] = pd.qcut(g["poverty_rate"].rank(method="first"), 3, labels=["low poverty", "middle", "high poverty"]).astype(str)
        g["majority_minority"] = np.where(g["minority_share"] > 0.5, "majority-minority", "white-majority or mixed"); DEMO.append(g)
        log(f"  {c}: {len(tr)} tracts → {len(g)} of {len(AREAS[c])} areas with census data")
        T = IV[c].merge(g[["area", "poverty_group", "majority_minority", "poverty_rate", "minority_share"]], on="area", how="inner")
        meths = [m[3:] for m in T.columns if m.startswith("lo|")]
        for gcol in ("poverty_group", "majority_minority"):
            for gv, gg in T.groupby(gcol):
                for meth in meths:
                    cov = (gg["y"] >= gg[f"lo|{meth}"]) & (gg["y"] <= gg[f"hi|{meth}"])
                    EQ_COV.append(dict(city=c, grouping=gcol, group=gv, method=meth, rows=len(gg), areas=gg["area"].nunique(), coverage=cov.mean(),
                                       mean_width=(gg[f"hi|{meth}"] - gg[f"lo|{meth}"]).mean(), mean_count=gg["y"].mean()))
        for meth in meths:                                             # per-area coverage vs demographics
            cov = ((T["y"] >= T[f"lo|{meth}"]) & (T["y"] <= T[f"hi|{meth}"])).groupby(T["area"]).mean().rename("coverage").reset_index().merge(g, on="area")
            rho_p = spearmanr(cov["coverage"], cov["poverty_rate"]); rho_m = spearmanr(cov["coverage"], cov["minority_share"])
            AREA_EQ.append(dict(city=c, method=meth, areas=len(cov), spearman_cov_poverty=rho_p[0], p_poverty=rho_p[1], spearman_cov_minority=rho_m[0], p_minority=rho_m[1],
                                sd_area_coverage=cov["coverage"].std()))
        # forecast skill over the 4-week average, and risk lists, by group (H = 1, seed 0)
        Tt = WK1[WK1["city"] == c][["area", "type", "target_week", "y", "ma4", SCOL]].merge(g[["area", "poverty_group", "majority_minority"]], on="area", how="inner")
        for gcol in ("poverty_group", "majority_minority"):
            for gv, gg in Tt.groupby(gcol):
                est, lo_, hi_ = block_boot(unit_dev(gg["y"].values, gg["ma4"].values + 0.05) - unit_dev(gg["y"].values, gg[SCOL].values), gg["target_week"].values)
                EQ_SKILL.append(dict(city=c, grouping=gcol, group=gv, deviance_model=pdev(gg["y"], gg[SCOL]), deviance_ma4=pdev(gg["y"], gg["ma4"] + 0.05),
                                     gain=est, ci_lo=lo_, ci_hi=hi_, mean_count=gg["y"].mean(), rel_mae=np.abs(gg["y"] - gg[SCOL]).mean() / max(gg["y"].mean(), 1e-6)))
        k10 = min(10, len(AREAS[c]) // 4)
        R_ = []
        for (w, k), gg in Tt.groupby(["target_week", "type"]):
            if (gg["y"] > 0).sum() < k10: continue
            top_true = set(gg.nlargest(k10, "y")["area"]); listed = gg.nlargest(k10, SCOL)
            for gv, l_ in listed.groupby("poverty_group"):
                R_.append(dict(group=gv, listed=len(l_), hits=int(l_["area"].isin(top_true).sum()), true_top=int(gg[gg["area"].isin(top_true)]["poverty_group"].eq(gv).sum())))
        if R_:
            R_ = pd.DataFrame(R_).groupby("group")[["listed", "hits", "true_top"]].sum().reset_index()
            R_["share_of_list_slots"] = R_["listed"] / R_["listed"].sum(); R_["share_of_true_top"] = R_["true_top"] / R_["true_top"].sum()
            R_["precision"] = R_["hits"] / R_["listed"]; EQ_RISK.append(R_.assign(city=c, list_size=k10))
    if DEMO: save_table(pd.concat(DEMO), "f01_area_demographics")
    EQ_COV = save_table(pd.DataFrame(EQ_COV), "f02_coverage_by_group"); EQ_SKILL = save_table(pd.DataFrame(EQ_SKILL), "f03_forecast_skill_by_group")
    if EQ_RISK: save_table(pd.concat(EQ_RISK), "f04_risk_list_attention_by_poverty_group")
    AREA_EQ = save_table(pd.DataFrame(AREA_EQ), "f05_area_coverage_vs_demographics")
    if len(EQ_COV):
        GAP = []
        for (c, gcol, meth), gg in EQ_COV.groupby(["city", "grouping", "method"]):
            v = gg.set_index("group")["coverage"]
            hi_lab, lo_lab = ("high poverty", "low poverty") if gcol == "poverty_group" else ("majority-minority", "white-majority or mixed")
            if hi_lab in v and lo_lab in v: GAP.append(dict(city=c, grouping=gcol, method=meth, coverage_disadvantaged=v[hi_lab], coverage_advantaged=v[lo_lab], gap=v[hi_lab] - v[lo_lab]))
        GAP = save_table(pd.DataFrame(GAP), "f06_coverage_gap_between_groups"); display(GAP.round(3))
        if len(GAP):
            fig, ax = plt.subplots(figsize=(10, 4.5)); gp = GAP[GAP["grouping"] == "poverty_group"].pivot_table(index="method", columns="city", values="gap") * 100
            gp.plot.barh(ax=ax); ax.axvline(0, color="k", lw=.8); ax.set_xlabel("coverage(high-poverty third) − coverage(low-poverty third), points")
            ax.set_title("Coverage gap between the poorest and the richest third of areas (H = 1, test)"); savefig("figF1_coverage_gap_poverty")
    else: log("  equity check: no city had census data – see the messages above (is the CENSUS_API_KEY secret switched on?)")
    display(EQ_SKILL.round(4)); display(AREA_EQ.round(3))

## Part I.2 Re-runs with more seeds
Order: ST-HSL, Graph WaveNet and MTGNN seeds 3–4 (about 1 hour together), then ST-SHN seed by seed in both cities (about 6.5 hours). Every finished run is saved at once in `predictions/patch_runs/` and skipped if the notebook is run again.

In [ ]:
def safe(name): return re.sub(r"[^A-Za-z0-9]+", "_", name)
def run_file(kind, c, s): return RUNS / f"{safe(kind)}_{c}_s{s}.npz"
EST_MIN = {("ST-HSL", "NYC"): 11, ("ST-HSL", "CHI"): 6, ("Graph WaveNet", "NYC"): 4, ("Graph WaveNet", "CHI"): 3, ("MTGNN", "NYC"): 3, ("MTGNN", "CHI"): 3,
           ("ST-SHN", "NYC"): 1.9 * STSHN_EPOCHS, ("ST-SHN", "CHI"): 1.25 * STSHN_EPOCHS}          # minutes per run on a T4 (from v1)

def run_stshn_budgets(c, seed, epochs, timeout_s):
    """Official ST-SHN code for `epochs` epochs; picks the epoch by validation and by test MAPE within the official budget and within the full budget."""
    d = REPOS / "ST-SHN-run"; t = time.time(); lf = LOGS / f"stshn_{c}_seed{seed}.log"; outp = str(d / f"out_{c}_s{seed}")
    for ext in ("_preds.npy", "_hist.json"): Path(outp + ext).unlink(missing_ok=True)       # never read a file left by an older run
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    cmd = [sys.executable, "HG_ST_labcode.py", "--data", c, "--task", "r", "--save_path", f"{c}_s{seed}", "--epoch", str(epochs)]
    env = {"STSHN_OUT": outp, "STSHN_SEED": str(seed), "TF_CPP_MIN_LOG_LEVEL": "2"}
    if DEVICE.type != "cuda": env["CUDA_VISIBLE_DEVICES"] = ""
    rc = run_logged(cmd, d, lf, timeout_s, env)
    if not Path(outp + "_preds.npy").exists(): log(f"  ST-SHN {c} seed {seed}: no epoch finished (exit {rc}); see {lf.name}"); return None
    P = np.load(outp + "_preds.npy"); H = json.load(open(outp + "_hist.json"))                # P: (epochs, cells, days, K)
    vm = np.array([h["val"]["MAPE"] for h in H]); tm = np.array([h["test"]["MAPE"] for h in H]); n = len(H)
    out = dict(val_mape=vm, test_mape=tm)
    info = dict(model="ST-SHN", city=c, seed=seed, epochs_run=n, epochs_planned=epochs, minutes=(time.time() - t) / 60, finished=(rc == 0))
    for tag, b in (("official", min(STSHN_OFFICIAL_EPOCHS, n)), ("long", n)):
        ev, et = int(np.argmin(vm[:b])), int(np.argmin(tm[:b]))
        out[f"val_{tag}"] = np.transpose(P[ev], (1, 0, 2)).reshape(-1); out[f"test_{tag}"] = np.transpose(P[et], (1, 0, 2)).reshape(-1)
        info[f"epoch_by_validation_{tag}"] = ev + 1; info[f"epoch_by_test_{tag}"] = et + 1; info[f"budget_{tag}"] = b
    log(f"  ST-SHN {c} seed {seed}: {n}/{epochs} epochs, {info['minutes']:.0f} min; epoch by validation / test: "
        f"first {info['budget_official']} epochs {info['epoch_by_validation_official']} / {info['epoch_by_test_official']}, all {n} epochs {info['epoch_by_validation_long']} / {info['epoch_by_test_long']}")
    return out, info

def bench_job(kind, c, s):
    f = run_file(kind, c, s)
    if f.exists(): log(f"  {kind} {c} seed {s}: already done, loaded"); return
    est = EST_MIN[(kind, c)]
    if time_left() * 60 < est * 1.3 + 25: log(f"  time budget: {kind} {c} seed {s} skipped (needs ~{est:.0f} min) – run the notebook again to finish it"); return
    yte = BENCH_DATA[c]["yte"]
    if kind == "ST-HSL":
        r = run_sthsl(c, s, 2 if FAST else 25, timeout_s=int(min(15 if FAST else 100, 60 * (time_left() - 0.5)) * 60))
        if r is None: return
        pt, pv, info = r; pt = pt.reshape(-1)
        if len(pt) != len(yte): log("  ST-HSL output does not line up with the test set; skipped"); return
        arrs = dict(pt=pt)
    elif kind == "ST-SHN":
        if not STSHN_OK: return
        r = run_stshn_budgets(c, s, STSHN_EPOCHS, timeout_s=int(min(30 if FAST else 6 * STSHN_EPOCHS, 60 * (time_left() - 0.5)) * 60))
        if r is None: return
        arrs, info = r
        if len(arrs["val_official"]) != len(yte): log("  ST-SHN output does not line up with the test set; skipped"); return
    else:
        pt, pv, info = train_graph_model(kind, c, s); arrs = dict(pt=pt.reshape(-1))
    np.savez_compressed(f, **arrs); json.dump(info, open(f.with_suffix(".json"), "w"), default=str)

with part("I.2 benchmark: forecasts, re-runs and protocols (v2)"):
    prep_sthsl()
    try: STSHN_DIR = prep_stshn(); STSHN_OK = True
    except Exception as e: log(f"ST-SHN not available: {e}"); STSHN_OK = False
    BENCH_DATA = {}; BPRED = {}
    for c in BENCH_CITIES:                                            # our forecaster (all 5 seeds; ~1 min per city) and the naive baselines
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
        trd, vad, ted = np.arange(30, ntr), np.arange(ntr, ntr + nva), np.arange(ntr + nva, T)
        (Xtr, ytr), (Xva, yva), (Xte, yte) = (bench_features(Xall, d) for d in (trd, vad, ted))
        cat = Xte["cat"].cat.codes.values; act = (Xall[:, :, :ntr] > 0).mean(2).reshape(N, K); tl = ((act > 0.2) & (act < 0.8)).reshape(-1)
        keep2 = np.tile(tl, nte) & (yte > 0)
        BENCH_DATA[c] = dict(yte=yte, yva=yva, cat=cat, K=K, keep2=keep2, nte=nte, N=N, Xall=Xall, ntr=ntr, Xte=Xte, Xva=Xva)
        P = BPRED.setdefault(c, {}); PV = BENCH_DATA[c].setdefault("val_preds", {}); t = time.time()
        for s in SEEDS_BENCH:
            ml = fit_lgb_pois(Xtr, ytr, Xva, yva, seed=s); mx = fit_xgb_pois(Xtr, ytr, Xva, yva, seed=s)
            P.setdefault("Ours: GBM-Poisson (LightGBM)", []).append(ml.predict(Xte)); PV.setdefault("Ours: GBM-Poisson (LightGBM)", []).append(ml.predict(Xva))
            P.setdefault("Ours: GBM-Poisson (XGBoost)", []).append(mx.predict(Xte)); PV.setdefault("Ours: GBM-Poisson (XGBoost)", []).append(mx.predict(Xva))
        vdev = {n: np.mean([pdev(yva, v) for v in PV[n]]) for n in ("Ours: GBM-Poisson (LightGBM)", "Ours: GBM-Poisson (XGBoost)")}
        BENCH_DATA[c]["ours"] = OURS = min(vdev, key=vdev.get)
        log(f"{c}: our forecasters ({len(SEEDS_BENCH)} seeds) in {time.time() - t:.0f}s; validation deviance {vdev} → {OURS}")
        P["Ours – protocol-aware (zero-truncated median)"] = [ztp_median(p) for p in P[OURS]]
        P["Naive: last day"] = [Xte["lag1"].values]; P["Naive: 30-day mean"] = [Xte["m30"].values]; P["Trivial: constant 1"] = [np.ones_like(yte)]

    # cheap runs first, then ST-SHN seed by seed in both cities (a time-out never leaves a city without runs)
    JOBS = [(k, c, s) for k in ("ST-HSL", "Graph WaveNet", "MTGNN") for s in NEW_SEEDS for c in BENCH_CITIES]
    JOBS += [("ST-SHN", c, s) for s in SEEDS_STSHN for c in BENCH_CITIES]
    for kind, c, s in JOBS:
        try: bench_job(kind, c, s)
        except Exception as e: log(f"  {kind} {c} seed {s} failed: {str(e)[:300]}"); open(LOGS / "errors.log", "a").write(f"{kind} {c} {s}\n{traceback.format_exc()}\n")
        refresh_zip()

### I.2b Merge v1 and v2, and score
ST-HSL / Graph WaveNet / MTGNN: v1 seeds 0–2 + v2 seeds 3–4. ST-SHN: v2 runs only; the main rows use the official 10-epoch budget, `b16` compares it with the 25-epoch budget.

In [ ]:
STSHN_VAL, STSHN_TEST = "ST-SHN (re-run, epoch chosen on validation)", "ST-SHN (re-run, as its code reports: best test epoch)"
with part("I.2b merge v1 and v2 predictions"):
    BRUNS, STSHN_LONG = [], []
    b05 = tab_v1("b05_rerun_log")
    for c in BENCH_CITIES:
        P = BPRED[c]; yte = BENCH_DATA[c]["yte"]
        V1 = load_npz(f"benchmark_{c}_test_predictions.npz")
        assert np.array_equal(V1["y"], yte), f"{c}: v1 test set differs from this run – the benchmark data changed"
        for kind in ("ST-HSL", "Graph WaveNet", "MTGNN"):
            name = "ST-HSL (re-run)" if kind == "ST-HSL" else f"{kind} (official model code, re-trained)"
            old = list(V1[safe(name)]) if safe(name) in V1.files else []
            P[name] = old[:3]                                         # v1 seeds 0, 1, 2 (in this order)
            if b05 is not None: BRUNS += b05[(b05["model"] == kind) & (b05["city"] == c)].assign(source="v1").to_dict("records")
            for s in NEW_SEEDS:
                f = run_file(kind, c, s)
                if f.exists(): P[name].append(np.load(f)["pt"]); BRUNS.append(json.load(open(f.with_suffix(".json"))) | dict(source="v2"))
            if not P[name]: del P[name]
            log(f"  {c} {name}: {len(P.get(name, []))} seeds")
        for s in SEEDS_STSHN:                                         # ST-SHN: v2 runs only (official budget = main rows)
            f = run_file("ST-SHN", c, s)
            if not f.exists(): continue
            z = np.load(f); info = json.load(open(f.with_suffix(".json")))
            P.setdefault(STSHN_VAL, []).append(z["val_official"]); P.setdefault(STSHN_TEST, []).append(z["test_official"])
            BRUNS.append(info | dict(source="v2", selected_epoch_validation=info["epoch_by_validation_official"], selected_epoch_test=info["epoch_by_test_official"]))
            for tag in ("official", "long"):
                for sel, key in (("validation", f"val_{tag}"), ("test (as the code reports)", f"test_{tag}")):
                    p = np.asarray(z[key], float); D = BENCH_DATA[c]
                    STSHN_LONG.append(dict(city=c, seed=s, budget_epochs=info[f"budget_{tag}"], epoch_selected_by=sel,
                                           selected_epoch=info[f"epoch_by_{'validation' if sel == 'validation' else 'test'}_{tag}"],
                                           P1_MAE=p_metrics(p, yte, D["cat"], D["K"], yte > 0)["MAE"], P1_MAPE=p_metrics(p, yte, D["cat"], D["K"], yte > 0)["MAPE"],
                                           P2_MAE=p_metrics(p, yte, D["cat"], D["K"], D["keep2"])["MAE"], P3_deviance_all=pdev(yte, np.maximum(p, 0.01))))
        log(f"  {c} ST-SHN: {len(P.get(STSHN_VAL, []))} seeds")
        out_npz = PRED / f"benchmark_{c}_test_predictions.npz"
        if out_npz.is_dir(): shutil.rmtree(out_npz)                   # v1 copy that Kaggle had unpacked into a folder
        np.savez_compressed(out_npz, y=yte, cat=BENCH_DATA[c]["cat"], **{safe(k): np.stack(v) for k, v in P.items()})
    BRUNS = save_table(pd.DataFrame(BRUNS), "b05_rerun_log"); display(BRUNS)
    STSHN_LONG = pd.DataFrame(STSHN_LONG)
    if len(STSHN_LONG):
        STSHN_LONG = save_table(STSHN_LONG, "b16_stshn_epoch_budget")
        display(STSHN_LONG.groupby(["city", "budget_epochs", "epoch_selected_by"])[["selected_epoch", "P1_MAE", "P1_MAPE", "P2_MAE", "P3_deviance_all"]].agg(["mean", "std"]).round(4))

In [ ]:
with part("I.2c benchmark scores under the three protocols (v2)"):
    # ---- scores
    ROWS = []
    for c in BENCH_CITIES:
        D = BENCH_DATA[c]; y, cat, K = D["yte"], D["cat"], D["K"]
        for name, plist in BPRED[c].items():
            for s, p in enumerate(plist):
                p = np.asarray(p, float)
                ROWS.append(dict(city=c, model=name, seed=s, **{f"P1_{k}": v for k, v in p_metrics(p, y, cat, K, y > 0).items()},
                                 **{f"P2_{k}": v for k, v in p_metrics(p, y, cat, K, D["keep2"]).items()}, **{f"P3_{k}": v for k, v in full_metrics(p, y, cat, K).items()}))
    BSC = save_table(pd.DataFrame(ROWS), "b06_benchmark_scores_per_seed")
    num = [c_ for c_ in BSC.columns if c_[:3] in ("P1_", "P2_", "P3_")]
    BMEAN = BSC.groupby(["city", "model"])[num].mean(); BSD = BSC.groupby(["city", "model"])[num].std(); BN = BSC.groupby(["city", "model"]).size().rename("seeds")
    BAGG = BMEAN.join(BSD.add_suffix("_sd")).join(BN).reset_index(); save_table(BAGG, "b07_benchmark_scores_mean_sd")

    # official-protocol table with the published numbers
    OFF = []
    for c in BENCH_CITIES:
        for name, v in PUBLISHED[c].items():
            OFF.append(dict(city=c, model=name, source="published (ST-HSL paper, Table III)", seeds=np.nan, **{f"MAE_{k}": v[2 * k] for k in range(4)}, **{f"MAPE_{k}": v[2 * k + 1] for k in range(4)}))
        for _, r in BAGG[BAGG["city"] == c].iterrows():
            OFF.append(dict(city=c, model=r["model"], source="this notebook (mean over seeds)", seeds=r["seeds"],
                            **{f"MAE_{k}": r[f"P1_MAE_{k}"] for k in range(4)}, **{f"MAPE_{k}": r[f"P1_MAPE_{k}"] for k in range(4)},
                            **{f"MAE_{k}_sd": r[f"P1_MAE_{k}_sd"] for k in range(4)}))
    OFF = pd.DataFrame(OFF); OFF["mean_MAE"] = OFF[[f"MAE_{k}" for k in range(4)]].mean(1); save_table(OFF, "b08_official_protocol_with_published")
    for c in BENCH_CITIES:
        ren = {f"MAE_{k}": f"{n} MAE" for k, n in enumerate(BENCH_CATS[c])} | {f"MAPE_{k}": f"{n} MAPE" for k, n in enumerate(BENCH_CATS[c])}
        print(f"\n{c} – official protocol P1 (crime days only)"); display(OFF[OFF["city"] == c].rename(columns=ren).drop(columns="city").sort_values("mean_MAE").round(4))
    FULLT = BAGG[["city", "model", "seeds", "P3_MAE_all", "P3_RMSE_all", "P3_deviance_all", "P3_deviance_all_sd", "P3_MAE_zero_days", "P3_MAE_crime_days"]]
    print("\nFull protocol P3 (all cell-days)"); display(FULLT.sort_values(["city", "P3_deviance_all"]).round(4))

### I.3 Reproduction gap, metric exploit and ranking stability (code as in v1)

In [ ]:
with part("I.3 benchmark: reproduction gap, exploit, rankings, intervals"):
    REPRO = []
    MAP_PUB = {"ST-HSL (re-run)": "ST-HSL", "ST-SHN (re-run, epoch chosen on validation)": "ST-SHN", "ST-SHN (re-run, as its code reports: best test epoch)": "ST-SHN",
               "Graph WaveNet (official model code, re-trained)": "GWN", "MTGNN (official model code, re-trained)": "MTGNN"}
    for c in BENCH_CITIES:
        for rerun, pubname in MAP_PUB.items():
            g = BSC[(BSC["city"] == c) & (BSC["model"] == rerun)]
            if g.empty: continue
            for k, n in enumerate(BENCH_CATS[c]):
                pub_mae, pub_mape = PUBLISHED[c][pubname][2 * k], PUBLISHED[c][pubname][2 * k + 1]
                m, sd = g[f"P1_MAE_{k}"].mean(), g[f"P1_MAE_{k}"].std()
                REPRO.append(dict(city=c, rerun=rerun, published_as=pubname, category=n, seeds=len(g), published_MAE=pub_mae, rerun_MAE=m, rerun_MAE_sd=sd,
                                  gap_MAE=m - pub_mae, gap_pct=100 * (m - pub_mae) / pub_mae, rerun_worse=m > pub_mae,
                                  published_MAPE=pub_mape, rerun_MAPE=g[f"P1_MAPE_{k}"].mean(), gap_MAPE=g[f"P1_MAPE_{k}"].mean() - pub_mape))
    REPRO = save_table(pd.DataFrame(REPRO), "b09_reproduction_gap"); display(REPRO.round(4))
    if not REPRO.empty:
        display(REPRO.groupby(["rerun"]).agg(cells=("gap_MAE", "size"), rerun_worse=("rerun_worse", "sum"), mean_gap_pct=("gap_pct", "mean")).round(2))
    # test-set selection optimism of ST-SHN
    SEL = BSC[BSC["model"].str.startswith("ST-SHN")].groupby(["city", "model"])[["P1_MAE", "P1_MAPE", "P2_MAE", "P2_MAPE", "P3_deviance_all"]].mean().reset_index()
    save_table(SEL, "b10_stshn_test_selection_optimism"); display(SEL.round(4))

    # metric exploit
    EXPL = []
    for c in BENCH_CITIES:
        pub = PUBLISHED[c]
        for name in ("Trivial: constant 1", "Ours – protocol-aware (zero-truncated median)", BENCH_DATA[c]["ours"]):
            g = BSC[(BSC["city"] == c) & (BSC["model"] == name)]
            for k, n in enumerate(BENCH_CATS[c]):
                mae, mape = g[f"P1_MAE_{k}"].mean(), g[f"P1_MAPE_{k}"].mean()
                EXPL.append(dict(city=c, model=name, category=n, MAE=mae, MAPE=mape, published_STHSL_MAE=pub["ST-HSL"][2 * k], published_STHSL_MAPE=pub["ST-HSL"][2 * k + 1],
                                 beats_STHSL_MAE=mae < pub["ST-HSL"][2 * k], beats_STHSL_MAPE=mape < pub["ST-HSL"][2 * k + 1],
                                 published_models_beaten_MAE=sum(v[2 * k] > mae for v in pub.values()), published_models_beaten_MAPE=sum(v[2 * k + 1] > mape for v in pub.values()),
                                 published_models=len(pub)))
    EXPL = save_table(pd.DataFrame(EXPL), "b11_metric_exploit"); display(EXPL.round(4))

    # rankings under the three protocols
    RANK, TAU = [], []
    for c in BENCH_CITIES:
        g = BAGG[BAGG["city"] == c].copy()
        g["P1_mean_MAE"] = g[[f"P1_MAE_{k}" for k in range(4)]].mean(1); g["P2_mean_MAE"] = g[[f"P2_MAE_{k}" for k in range(4)]].mean(1)
        cols = {"P1 official ST-HSL (mean MAE, crime days)": "P1_mean_MAE", "P2 official ST-SHN (mean MAE, subset)": "P2_mean_MAE",
                "P3 full (Poisson deviance, all cell-days)": "P3_deviance_all", "P3 full (MAE, all cell-days)": "P3_MAE_all"}
        for lab, col in cols.items(): g[f"rank: {lab}"] = g[col].rank(method="min")
        RANK.append(g[["city", "model", "seeds"] + list(cols.values()) + [f"rank: {l}" for l in cols]])
        for a_, b_ in (("P1_mean_MAE", "P3_deviance_all"), ("P1_mean_MAE", "P3_MAE_all"), ("P2_mean_MAE", "P3_deviance_all"), ("P1_mean_MAE", "P2_mean_MAE")):
            tau, pv = kendalltau(g[a_], g[b_])
            TAU.append(dict(city=c, protocol_a=a_, protocol_b=b_, kendall_tau=tau, p_value=pv, best_a=g.loc[g[a_].idxmin(), "model"], best_b=g.loc[g[b_].idxmin(), "model"],
                            models=len(g)))
    RANK = save_table(pd.concat(RANK), "b12_rankings_by_protocol"); TAU = save_table(pd.DataFrame(TAU), "b13_ranking_agreement")
    display(RANK.sort_values(["city", "P3_deviance_all"]).round(4)); display(TAU.round(3))

    # simple vs deep on the full protocol, seed by seed (A4)
    A4 = []
    for c in BENCH_CITIES:
        ours = BSC[(BSC["city"] == c) & (BSC["model"] == BENCH_DATA[c]["ours"])]["P3_deviance_all"].values
        for m in [x for x in BSC["model"].unique() if x.startswith(("ST-HSL", "ST-SHN (re-run, epoch chosen", "Graph WaveNet", "MTGNN"))]:
            dv = BSC[(BSC["city"] == c) & (BSC["model"] == m)]["P3_deviance_all"].values
            if len(dv): A4.append(dict(city=c, deep_model=m, seeds=len(dv), deep_deviance_min=dv.min(), deep_deviance_mean=dv.mean(),
                                       ours_deviance_max=ours.max(), ours_deviance_mean=ours.mean(), ours_better_every_seed=bool(ours.max() < dv.min())))
    A4 = save_table(pd.DataFrame(A4), "b14_simple_vs_deep_full_protocol"); display(A4.round(4))

    # 90 % intervals on the benchmark (calibrated on the 30 validation days, our forecaster, seed 0)
    BCOV = []
    for c in BENCH_CITIES:
        D = BENCH_DATA[c]; mu = np.asarray(BPRED[c][D["ours"]][0]); mv = np.asarray(D["val_preds"][D["ours"]][0]); y, yv = D["yte"], D["yva"]
        vol = pd.Series(D["Xall"][:, :, :D["ntr"]].mean(2).reshape(-1)); vq = pd.qcut(vol.rank(method="first"), 5, labels=False).values + 1
        vq_rows = np.tile(vq, D["nte"]); sc = norm_score(yv, mv); edges = mondrian_edges(mv); bv = np.digitize(mv, edges)
        qb = np.array([conformal_q(sc[bv == i]) for i in range(len(edges) + 1)]); r_nb = nb_size(mv, yv)
        meths = {"Poisson": parametric_interval(mu, "Poisson"), "Neg. binomial": parametric_interval(mu, "NB", r_nb),
                 "Split conformal": int_from_q(mu, conformal_q(sc)), "Mondrian conformal": int_from_q(mu, qb[np.digitize(mu, edges)])}
        for meth, (lo, hi) in meths.items(): BCOV.append(dict(city=c, method=meth, **coverage_report(y, lo, hi, vq_rows)))
    BCOV = save_table(pd.DataFrame(BCOV), "b15_benchmark_interval_coverage"); display(BCOV.round(3))

    # figures
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    for i, c in enumerate(BENCH_CITIES):
        d = OFF[OFF["city"] == c].sort_values("mean_MAE")
        col = ["#c0392b" if m.startswith("Ours:") else "#e67e22" if ("constant" in m or "protocol-aware" in m) else "#27ae60" if ("re-run" in m or "re-trained" in m)
               else "#7f8c8d" if s_.startswith("published") else "#2980b9" for m, s_ in zip(d["model"], d["source"])]
        ax[i].barh(d["model"] + np.where(d["source"].str.startswith("published"), " [pub.]", ""), d["mean_MAE"], color=col); ax[i].invert_yaxis()
        ax[i].set_title(f"{c}-Crimes: official protocol (crime days only), mean MAE over 4 categories\ngrey published · green re-run · red ours · orange metric exploit · blue naive", fontsize=9)
        ax[i].tick_params(axis="y", labelsize=7)
    savefig("figB1_official_protocol_published_vs_rerun")
    if not REPRO.empty:
        fig, ax = plt.subplots(figsize=(11, 4.5)); g = REPRO.copy(); g["label"] = g["city"] + " " + g["category"]
        for j, (m, gg) in enumerate(g.groupby("rerun")): ax.scatter(gg["gap_pct"], gg["label"], label=m, s=40, marker="osD^v"[j % 5])
        ax.axvline(0, color="k", lw=.8); ax.set_xlabel("re-run minus published official MAE (%)"); ax.legend(fontsize=7); ax.set_title("Reproduction gap (positive = re-run worse than published)")
        savefig("figB2_reproduction_gap")
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for i, c in enumerate(BENCH_CITIES):
        g = RANK[RANK["city"] == c]
        for _, r in g.iterrows():
            ax[i].plot([0, 1], [r["rank: P1 official ST-HSL (mean MAE, crime days)"], r["rank: P3 full (Poisson deviance, all cell-days)"]], marker="o")
            ax[i].text(1.03, r["rank: P3 full (Poisson deviance, all cell-days)"], r["model"][:45], fontsize=7, va="center")
        ax[i].set_xticks([0, 1]); ax[i].set_xticklabels(["official (crime days)", "full (deviance)"]); ax[i].invert_yaxis(); ax[i].set_xlim(-.1, 1.9)
        ax[i].set_ylabel("rank (1 = best)"); ax[i].set_title(f"{c}: model ranks change with the protocol")
    savefig("figB3_rank_change_by_protocol")

### I.4 Reproduction gap against seed noise
With 5 seeds the gap of each city × category cell gets a 95 % t-interval: *re-run worse* only if the whole interval lies above the published number.

In [ ]:
from scipy.stats import t as _t
with part("I.4 reproduction gap against seed noise (v2)"):
    # is the re-run mean really above the published number? one-sample 95 % t-interval over seeds, per city × category
    GAPCI = []
    for c in BENCH_CITIES:
        for rerun, pubname in MAP_PUB.items():
            g = BSC[(BSC["city"] == c) & (BSC["model"] == rerun)]
            if len(g) < 2: continue
            for k, n in enumerate(BENCH_CATS[c]):
                v = g[f"P1_MAE_{k}"].values; pub = PUBLISHED[c][pubname][2 * k]; m, sd = v.mean(), v.std(ddof=1); h = _t.ppf(0.975, len(v) - 1) * sd / np.sqrt(len(v))
                GAPCI.append(dict(city=c, rerun=rerun, category=n, seeds=len(v), published_MAE=pub, rerun_MAE=m, rerun_MAE_sd=sd, gap_MAE=m - pub,
                                  gap_ci_lo=m - h - pub, gap_ci_hi=m + h - pub, gap_pct=100 * (m - pub) / pub,
                                  verdict="re-run worse" if m - h > pub else ("re-run better" if m + h < pub else "within seed noise"),
                                  seeds_worse_than_published=int((v > pub).sum())))
    GAPCI = save_table(pd.DataFrame(GAPCI), "b09b_reproduction_gap_ci"); display(GAPCI.round(4))
    if len(GAPCI): display(GAPCI.groupby(["city", "rerun"])["verdict"].value_counts().unstack(fill_value=0))

## Summary: hypothesis verdicts
Pre-registered hypotheses A1–A8 with v1's thresholds, plus three rows marked *post hoc* (A1b, A6b, A7b) that follow from the v1 results and must be reported as such in the paper.

In [ ]:
# tables of Parts II–IV come from v1 (unchanged in this patch)
SKS, COV, ROLL_COV, ROLL_SKILL, DS = (tab_v1(n) for n in ("c07_skill_summary", "d01_interval_coverage_main_split", "e01_rolling_interval_coverage",
                                                          "e02_rolling_forecast_skill", "c02_panel_summary"))
RATE_MODELS = {"lgbm": "LightGBM-Poisson", "xgb": "XGBoost-Poisson", "glm": "Poisson GLM", "gru": "GRU (global)", "stgru": "ST-GRU (graph diffusion input)"}
O3_SEL = "lgbm"
def g_(name):
    v = globals().get(name); return v if isinstance(v, pd.DataFrame) and len(v) else None
def verdict(ok, partly=False): return "supported" if ok else ("partly supported" if partly else "not supported")
HYP = []
with part("VI. hypothesis verdicts and summary"):
    R_ = g_("REPRO")
    if R_ is not None:
        ev = []; oks = []
        for m in ("ST-HSL (re-run)", "ST-SHN (re-run, epoch chosen on validation)"):
            gg = R_[R_["rerun"] == m]
            if len(gg): oks.append(gg["rerun_worse"].sum() >= 6); ev.append(f"{m}: re-run worse in {int(gg['rerun_worse'].sum())}/{len(gg)} cells, mean gap {gg['gap_pct'].mean():+.1f} % ({int(gg['seeds'].max())} seeds)")
        HYP.append(dict(hypothesis="A1", claim="re-runs of ST-HSL and ST-SHN are worse than published in ≥ 6/8 cells",
                        verdict=verdict(len(oks) == 2 and all(oks), any(oks)) if oks else "not evaluated", evidence="; ".join(ev) if ev else "no ST-HSL / ST-SHN re-run finished"))
    else: HYP.append(dict(hypothesis="A1", claim="reproduction gap", verdict="not evaluated", evidence="benchmark part did not run"))
    G_ = g_("GAPCI")
    if G_ is not None:                                                # post hoc: the gap measured against seed noise
        gg = G_[G_["rerun"] == "ST-HSL (re-run)"]; w = gg.groupby("city")["verdict"].apply(lambda s: int((s == "re-run worse").sum()))
        HYP.append(dict(hypothesis="A1b (post hoc)", claim="ST-HSL re-run worse than published beyond seed noise (95 % t-interval above 0) in ≥ 6/8 cells",
                        verdict=verdict((gg["verdict"] == "re-run worse").sum() >= 6, (gg["verdict"] == "re-run worse").any()),
                        evidence="; ".join(f"{c}: {w.get(c, 0)}/4 cells" for c in BENCH_CITIES) + f" ({int(gg['seeds'].max()) if len(gg) else 0} seeds)"))
    E_ = g_("EXPL")
    if E_ is not None:
        c1 = E_[E_["model"] == "Trivial: constant 1"].groupby("city")["beats_STHSL_MAPE"].sum(); z = E_[E_["model"].str.contains("protocol-aware")].groupby("city")["beats_STHSL_MAE"].sum()
        ok = (c1 > 0).all() and (z > 0).all() and len(c1) == 2
        HYP.append(dict(hypothesis="A2", claim="a constant 1 beats published ST-HSL MAPE, and the protocol-aware forecast beats its MAE, in ≥ 1 category per city", verdict=verdict(ok, (c1 > 0).any()),
                        evidence="; ".join(f"{c}: constant-1 beats MAPE in {int(c1.get(c, 0))}/4, protocol-aware beats MAE in {int(z.get(c, 0))}/4" for c in BENCH_CITIES)))
    T_ = g_("TAU")
    if T_ is not None:
        t1 = T_[(T_["protocol_a"] == "P1_mean_MAE") & (T_["protocol_b"] == "P3_deviance_all")]
        ok = ((t1["kendall_tau"] < 0.8) | (t1["best_a"] != t1["best_b"])).any()
        HYP.append(dict(hypothesis="A3", claim="rankings disagree between the official and the full protocol (τ < 0.8 or a different winner)", verdict=verdict(ok),
                        evidence="; ".join(f"{r.city}: τ = {r.kendall_tau:.2f}, best official = {r.best_a}, best full = {r.best_b}" for r in t1.itertuples())))
    A4_ = g_("A4")
    if A4_ is not None:
        HYP.append(dict(hypothesis="A4", claim="our forecaster has lower full-protocol deviance than every re-run deep model, in every seed", verdict=verdict(A4_["ours_better_every_seed"].all(), A4_["ours_better_every_seed"].any()),
                        evidence="; ".join(f"{r.city} vs {r.deep_model.split(' (')[0]}: {r.ours_deviance_mean:.3f} vs {r.deep_deviance_mean:.3f} ({r.seeds} seeds)" for r in A4_.itertuples())))
    S_ = g_("SKS")
    if S_ is not None:
        nv = S_[S_["baseline_family"] == "naive/statistical"]
        HYP.append(dict(hypothesis="A5", claim="the selected forecaster beats every naive/statistical baseline at H = 1–4 in all cities and seeds", verdict=verdict(nv["significant_all_seeds"].all(), nv["significant_all_seeds"].mean() > 0.5),
                        evidence=f"{int(nv['significant_all_seeds'].sum())}/{len(nv)} horizon × city × baseline cells significant in every seed; selected = {RATE_MODELS[O3_SEL]} (v1)"))
    C_ = g_("COV")
    if C_ is not None:
        c0 = C_[C_["seed"] == C_.groupby(["H", "city"])["seed"].transform("min")]
        mon = c0[c0["method"] == "Mondrian CP"]; others = {m: c0[c0["method"] == m] for m in ("Split conformal (CP)", "Poisson", "Neg. binomial")}
        ok_m = (mon["max_abs_dev_from_90"] <= 0.03).all(); ok_o = all((g["max_abs_dev_from_90"] > 0.03).any() for g in others.values())
        HYP.append(dict(hypothesis="A6", claim="Mondrian keeps every volume quintile within 3 points of 90 % (all cities, horizons); CP, Poisson, NB do not", verdict=verdict(ok_m and ok_o, ok_m),
                        evidence=f"Mondrian worst deviation {100 * mon['max_abs_dev_from_90'].max():.1f} pts; " + "; ".join(f"{m} worst {100 * g['max_abs_dev_from_90'].max():.1f} pts" for m, g in others.items()) + " (v1)"))
        # post hoc: separate under-coverage (the harmful direction) from over-coverage
        qc = [f"cov_Q{q}" for q in range(1, 6)]
        def under(g): return float((0.9 - g[qc]).max(axis=1).max())
        def over(g): return float((g[qc] - 0.9).max(axis=1).max())
        rows = {m: c0[c0["method"] == m] for m in ("Mondrian CP", "Mondrian-ACI", "CQR", "Split conformal (CP)", "Poisson", "Neg. binomial")}
        rows = {m: g for m, g in rows.items() if len(g)}
        HYP.append(dict(hypothesis="A6b (post hoc)", claim="Mondrian never under-covers a volume quintile by more than 3 points (all cities, horizons); deviations above 3 points are over-coverage only",
                        verdict=verdict(under(rows["Mondrian CP"]) <= 0.03), evidence="; ".join(f"{m}: worst under-coverage {100 * under(g):.1f}, worst over-coverage {100 * over(g):.1f} pts" for m, g in rows.items()) + " (v1)"))
    RC_ = g_("ROLL_COV")
    if RC_ is not None:
        mon = RC_[RC_["method"] == "Mondrian CP"]; ok1 = (mon["max_abs_dev_from_90"] <= 0.03).all()
        a20 = RC_[(RC_["test_year"] == 2020) & RC_["method"].isin(["ACI", "Mondrian-ACI"])]
        ok2 = len(a20) > 0 and (a20.groupby("city")["coverage"].apply(lambda s: (s - 0.9).abs().min()) <= 0.02).all()
        HYP.append(dict(hypothesis="A7", claim="A6 holds in every rolling test year; in 2020 ACI brings coverage within 2 points of 90 %", verdict=verdict(ok1 and ok2, ok1 or ok2),
                        evidence=f"Mondrian worst deviation over folds {100 * mon['max_abs_dev_from_90'].max():.1f} pts (years {[int(v) for v in sorted(mon['test_year'].unique())]}); "
                                 + "2020 ACI coverage: " + ", ".join(f"{r.city} {r.method} {100 * r.coverage:.1f} %" for r in a20.itertuples()) + " (v1)"))
        ma = RC_[RC_["method"] == "Mondrian-ACI"]
        if len(ma):
            w = ma.groupby("city")["max_abs_dev_from_90"].max()
            HYP.append(dict(hypothesis="A7b (post hoc)", claim="Mondrian-ACI keeps every volume quintile within 3 points of 90 % in every rolling test year (incl. 2020)",
                            verdict=verdict((w <= 0.03).all(), (w <= 0.03).any()), evidence="worst quintile deviation over folds: " + ", ".join(f"{c} {100 * v:.1f} pts" for c, v in w.items()) + " (v1)"))
    GP_ = g_("GAP")
    if GP_ is not None:
        gp = GP_[GP_["grouping"] == "poverty_group"].pivot_table(index="city", columns="method", values="gap")
        if {"Mondrian CP", "Split conformal (CP)"} <= set(gp.columns):
            better = gp["Mondrian CP"].abs() < gp["Split conformal (CP)"].abs()
            extra = [m for m in ("Mondrian-ACI",) if m in gp.columns]
            HYP.append(dict(hypothesis="A8", claim="Mondrian has a smaller high- vs low-poverty coverage gap than CP in every city", verdict=verdict(better.all(), better.any()),
                            evidence="; ".join(f"{c}: Mondrian {100 * gp.loc[c, 'Mondrian CP']:+.1f} vs CP {100 * gp.loc[c, 'Split conformal (CP)']:+.1f} pts"
                                               + "".join(f", {m} {100 * gp.loc[c, m]:+.1f}" for m in extra) for c in gp.index)))
    if not any(h["hypothesis"] == "A8" for h in HYP): HYP.append(dict(hypothesis="A8", claim="equity of coverage", verdict="not evaluated", evidence="census data or intervals unavailable"))
    HYP = save_table(pd.DataFrame(HYP), "z01_hypothesis_verdicts"); display(HYP)

    def md(df, n=60):
        try: return df.head(n).to_markdown(index=False, floatfmt=".4f")
        except Exception: return df.head(n).to_string(index=False)
    STATUS["VI. hypothesis verdicts and summary"] = "ok"
    ALLSTAT = [dict(part=k, run="v1", status=v) for k, v in STATUS_V1.items()] + [dict(part=k, run="v2 (patch)", status=v) for k, v in STATUS.items()]
    lines = [f"# Option A experiments – results summary (v1 + v2 patch)", f"Generated {pd.Timestamp.now():%Y-%m-%d %H:%M}, patch run time {hours_used():.2f} h, FAST = {FAST}.",
             "Parts II–IV (weekly forecasts, intervals, rolling origins) are from v1; the benchmark tables (b05–b16), the equity tables (f01–f06) and the verdicts are recomputed in v2.", "",
             "## Part status", md(pd.DataFrame(ALLSTAT)), "", "## Hypotheses", md(HYP), ""]
    for title, name in [("Benchmark: official protocol vs published (mean over seeds)", "OFF"), ("Benchmark: full protocol", "FULLT"),
                        ("Benchmark: reproduction gap", "REPRO"), ("Benchmark: reproduction gap vs seed noise (95 % t-interval)", "GAPCI"),
                        ("Benchmark: ST-SHN epoch budget (official 10 vs longer)", "STSHN_LONG"), ("Benchmark: ranking agreement", "TAU"), ("Benchmark: protocol audit", "AUDIT"),
                        ("Benchmark: re-run log", "BRUNS"), ("Weekly panels (v1)", "DS"), ("Forecast skill summary (v1)", "SKS"), ("Interval coverage, main split (v1)", "COV"),
                        ("Rolling-origin coverage (v1)", "ROLL_COV"), ("Rolling-origin skill (v1)", "ROLL_SKILL"), ("Coverage gap between groups", "GAP"),
                        ("Forecast skill by group", "EQ_SKILL"), ("Area coverage vs demographics", "AREA_EQ")]:
        d = g_(name)
        if d is not None: lines += [f"## {title}", md(d, 120), ""]
    (OUT / "results_summary.md").write_text("\n".join(lines))
    json.dump(dict(status=STATUS, status_v1=STATUS_V1, hours=hours_used(), versions=VERSIONS, hypotheses=HYP.to_dict("records")), open(OUT / "metrics_summary.json", "w"), indent=1, default=str)
    log("results_summary.md written")
print(f"Finished in {hours_used():.2f} h. Download /kaggle/working/optionA_outputs_v2.zip from the Output tab.")